# 02 — reservoir sanity
Drive the connectome reservoir with the dual 4/4 + riff-cycle + form input and check the activity is neither dead nor saturated (raster + rank), then fit the readout and diff against the transcription.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')
import numpy as np, matplotlib.pyplot as plt
from fruit_fly_djent import connectome
from fruit_fly_djent.model import ComposerModel, targets_for
from fruit_fly_djent.reservoir import sanity_report
from fruit_fly_djent.transcription import load_song
song = load_song(); neurons, edges = connectome.load_subgraph()
model = ComposerModel.build(neurons, edges, song)
X, U = model.drive(song)
print(sanity_report(X))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax[0].imshow(X[:1024, :400].T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1); ax[0].set_title('reservoir states (first 16 bars, 400 neurons)')
ax[1].plot(U[:1024, :8]); ax[1].set_title('drum stream inputs'); plt.tight_layout()

In [ ]:
from fruit_fly_djent.train_supervised import ridge_fit
from fruit_fly_djent.reservoir import features
from fruit_fly_djent.sonify import decode_notes, diff_notes
Y, layout = targets_for(song)
F = features(X, U); Wo = ridge_fit(F, Y, 2e-4)
notes = decode_notes(F @ Wo, layout)
diff_notes(notes, song.notes, bpm=song.bpm)